# Fine-tune FunctionGemma for your Home Mind

This trains Google's FunctionGemma (270M parameters) to turn requests like *"dim the den lamp to 40"* into one of Home Mind's control tools, using **your house's own device names**. The result is small enough to run on an old laptop's CPU.

Based on Google's [FunctionGemma fine-tuning guide](https://ai.google.dev/gemma/docs/functiongemma/finetuning-with-functiongemma) (TRL `SFTTrainer`, full fine-tune).

**Before you start**
1. *Runtime → Change runtime type*: **L4** or **A100** if you have Colab Pro (included with Google AI Pro), otherwise **T4** (free). The notebook adapts to whichever you get.
2. Accept the Gemma terms at https://huggingface.co/google/functiongemma-270m-it and make a Hugging Face read token.
3. Download your training data from Home Mind (admin token): `GET /api/control_model/dataset?part=train` and `?part=test` (add `&paraphrase=1` to the train link for more varied wording from your general model). You get `home_mind_control_train.jsonl` and `home_mind_control_test.jsonl`. They contain your device names, nothing else from your memory.

Training takes about 10–30 minutes on a T4 depending on how many devices you have.

**Sleep and workouts too (0.28):** the training file now includes the household's phrasebook (odd ways of saying sleep and workouts), teaching two more tools, `log_sleep` and `log_workout`, whose argument is the plain log line ("slept from 1 am to 6:20 am", "squat 3x5 at 185 lb"). Once trained, Home Mind asks this model before a big one when a message about sleep or exercise isn't read by the built-in rules; its line goes through the same checks (it must parse, and no number that wasn't said). Download a fresh training file from Home Mind before training so it has the latest phrasebook.

In [ ]:
%pip install -q "torch" "transformers>=4.57" "datasets" "accelerate" "trl>=0.24" "sentencepiece" "protobuf"


In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > L4, A100 or T4"
GPU = torch.cuda.get_device_name(0)
MEM = torch.cuda.get_device_properties(0).total_memory / 2**30
BF16 = torch.cuda.is_bf16_supported()
# a 270M model is small: the batch size is about speed, not fitting in memory
BATCH = 32 if MEM >= 35 else 16 if MEM >= 20 else 8
print(f"{GPU}, {MEM:.0f} GB, bf16={BF16}, batch size {BATCH}")


In [ ]:
from huggingface_hub import login
login()  # paste your Hugging Face read token (Gemma terms must be accepted)


## 1. Upload your data

In [ ]:
import json
from google.colab import files
from datasets import Dataset, DatasetDict, load_dataset
up = files.upload()  # pick home_mind_control_bundle.json (1-click) OR both .jsonl files
bundle_name = next((k for k in up if k.endswith('.json')), None)
if bundle_name:
    b = json.loads(up[bundle_name].decode('utf-8'))
    ds = DatasetDict({'train': Dataset.from_list(b['train']), 'test': Dataset.from_list(b['test'])})
else:
    ds = load_dataset('json', data_files={'train': 'home_mind_control_train.jsonl', 'test': 'home_mind_control_test.jsonl'})
print(ds)
print(ds['train'][0]['messages'])


### Optional: more varied wording from Gemini (teacher)

Rewrites a sample of your training requests the way people actually talk, and keeps a rewrite only when your device name and numbers survive, so the answer stays right. Needs a Gemini API key from [Google AI Studio](https://aistudio.google.com/apikey) (free tier is enough), saved in Colab's **Secrets** (key icon on the left) as `GEMINI_API_KEY`. Only device names and request wording are sent.

In [ ]:
USE_GEMINI_TEACHER = False   # set True to use it
GEMINI_MODEL = "gemini-flash-latest"
REWRITES, SAMPLE = 3, 300

if USE_GEMINI_TEACHER:
    %pip install -q google-genai
    import json, random, re
    from google import genai
    from google.colab import userdata
    from datasets import concatenate_datasets, Dataset
    client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))
    rows = [r for r in ds["train"] if r["messages"][-1]["tool_calls"][0]["function"]["name"] != "not_a_command"]
    random.Random(7).shuffle(rows)
    extra = []
    for r in rows[:SAMPLE]:
        user = r["messages"][-2]["content"]
        call = r["messages"][-1]["tool_calls"][0]["function"]
        args = call["arguments"]
        prompt = (f"Rewrite this smart-home request in {REWRITES} different natural ways a person might say it out loud. "
                  "Keep the device words exactly as written and keep every number. Reply with a JSON list of strings only.\n\n"
                  + user)
        try:
            text = client.models.generate_content(model=GEMINI_MODEL, contents=prompt).text
            alts = json.loads(re.search(r"\[.*\]", text, re.S).group(0))
        except Exception as e:
            print("skip:", e); continue
        for a in alts:
            if not isinstance(a, str) or str(args.get("device", "")).lower() not in a.lower():
                continue
            if any(str(v) not in a for k, v in args.items() if k != "device" and isinstance(v, int)):
                continue
            if "color" in args and str(args["color"]).lower() not in a.lower():
                continue
            m = [dict(x) for x in r["messages"][:-2]] + [{"role": "user", "content": a}, r["messages"][-1]]
            extra.append({"messages": m, "tools": r["tools"]})
    print(f"kept {len(extra)} rewrites")
    if extra:
        ds["train"] = concatenate_datasets([ds["train"], Dataset.from_list(extra)]).shuffle(seed=7)
    print(ds)


## 2. Load FunctionGemma and check the prompt format

In [ ]:
import torch, json, re
from transformers import AutoModelForCausalLM, AutoTokenizer
base_model = "google/functiongemma-270m-it"
model = AutoModelForCausalLM.from_pretrained(base_model, dtype="auto", device_map="auto", attn_implementation="eager")
tokenizer = AutoTokenizer.from_pretrained(base_model)
ex = ds["train"][0]
print(tokenizer.apply_chat_template(ex["messages"], tools=ex["tools"], tokenize=False)[-600:])


## 3. Score before training

Exact match on the held-out test set: right tool and right arguments (device words, numbers, colors). The test set uses wordings and some devices that never appear in training.

In [ ]:
CALL = re.compile(r"call:(\w+)\{(.*?)\}", re.S)
ARG = re.compile(r"(\w+):(?:<escape>(.*?)<escape>|([^,}]*))", re.S)

def predict(messages, tools):
    inputs = tokenizer.apply_chat_template(messages, tools=tools, add_generation_prompt=True,
                                           return_dict=True, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=64, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    text = tokenizer.decode(out[0][len(inputs["input_ids"][0]):], skip_special_tokens=False)
    m = CALL.search(text)
    if not m:
        return None, {}, text
    args = {}
    for a in ARG.finditer(m.group(2)):
        v = (a.group(2) if a.group(2) is not None else a.group(3) or "").strip()
        args[a.group(1)] = int(v) if v.isdigit() else v
    return m.group(1), args, text

def words(s):
    return set(w for w in re.findall(r"[a-z0-9]+", str(s).lower()) if w not in ("the", "a", "my", "our", "in", "of"))

def evaluate(rows, show=5):
    ok, bad = 0, []
    for r in rows:
        want = r["messages"][-1]["tool_calls"][0]["function"]
        tool, args, text = predict(r["messages"][:-1], r["tools"])
        good = tool == want["name"]
        if good and tool != "not_a_command":
            for k, v in want["arguments"].items():
                good &= (words(args.get(k)) == words(v)) if k == "device" else str(args.get(k)).lower() == str(v).lower()
        ok += good
        if not good and len(bad) < show:
            bad.append((r["messages"][-2]["content"], want, text[:120]))
    for b in bad:
        print("MISS", b)
    return round(ok / len(rows), 3)

print("before:", evaluate(ds["test"]))


## 4. Train

In [ ]:
from trl import SFTConfig, SFTTrainer
args = SFTConfig(
    output_dir="functiongemma-home-mind",
    max_length=2048,            # the tool list is long: 13 tools
    packing=False,
    num_train_epochs=4,
    per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=1,
    learning_rate=5e-5,
    lr_scheduler_type="constant",
    optim="adamw_torch_fused",
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="no",
    bf16=BF16,
    fp16=not BF16,
    report_to="none",
)
trainer = SFTTrainer(model=model, args=args, train_dataset=ds["train"], eval_dataset=ds["test"],
                     processing_class=tokenizer)
trainer.train()
model = trainer.model.eval()
print("after:", evaluate(ds["test"]))


If *after* is below about 0.9: turn on the Gemini teacher above, or raise `num_train_epochs` to 6, and train again. On an L4 or A100 a run takes a few minutes, so trying a few settings is cheap.

## 5. Save and convert for llama.cpp / Ollama (GGUF)

In [ ]:
trainer.save_model("functiongemma-home-mind")
tokenizer.save_pretrained("functiongemma-home-mind")
!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp
%pip install -q -r llama.cpp/requirements/requirements-convert_hf_to_gguf.txt
!python llama.cpp/convert_hf_to_gguf.py functiongemma-home-mind --outtype q8_0 --outfile home-mind-control.gguf
!ls -lh home-mind-control.gguf
from google.colab import files
files.download("home-mind-control.gguf")


## 6. Run it at home

The file is about 300 MB. On Home Assistant OS:

1. Copy `home-mind-control.gguf` into the **share** folder as `home_mind/home-mind-control.gguf` (Samba share or File editor add-on).
2. Install and start the **Home Mind Control Model** add-on from the same repository. It runs llama.cpp's server on the CPU, reachable only inside Home Assistant. Note its **Hostname** on the Info page.
3. In Home Mind's options:

```yaml
models:
  - name: control
    type: openai
    url: http://HOSTNAME:8081/v1
control_model: control
```

Home Mind sends each house request to it first. It only *proposes* a tool call: the device name is matched against your control list, locks and garage doors still need your "yes", and anything it marks `not_a_command` (questions, several actions at once) goes to your normal model. Retrain whenever you add or rename devices, replace the file and restart the add-on.